# 7단계 — 오분류 클래스에 집중한 스태킹 메타모델 학습 가중치 실험

**6단계 결론:** B/O 보조 분류기는 제한적인 개선을 보였지만, P/O 보조 분류기는 Pastry F1을 낮췄습니다. 이번에는 보조 분류기를 늘리지 않고 **원래 ET+LightGBM 스태킹의 최종 LogisticRegression 학습에 클래스별 가중치**를 적용합니다.

- 외부 평가: `StratifiedKFold(n_splits=3, shuffle=True, random_state=2069)`
- 외부 학습 Fold에서 5-Fold OOF 베이스 확률을 만든 뒤 메타모델 학습 → 외부 검증에서 평가(누수 방지)
- **동일 외부 Fold에서** 기존 LR / Pastry 가중치 1.25 / Pastry 1.5 / Bumps+Pastry 가중치를 비교합니다.
- 기존 B/O 보조 모델은 효과가 작으므로 여기서는 제외합니다. 이미 좋은 클래스를 훼손하지 않는지 F1도 확인합니다.
- 이 데이터는 기존과 동일한 1,552행 개발 데이터입니다. **랜덤시드를 변경해도 독립 테스트가 되는 것은 아닙니다.** 389행 홀드아웃은 모델 선택에 사용하지 마세요.
- 목표 0.835는 외부 Fold Macro F1 평균으로 확인하지만, 반복적으로 탐색한 성능을 최종 일반화 성능으로 해석하지 않습니다.

**필요 파일:** `Faults.NNA`, `Faults27x7_var`. 이 노트북과 같은 폴더에 저장하고, 모든 셀을 위에서부터 실행하세요. 이전 노트북 재실행은 필요하지 않습니다.

In [1]:
# 필요한 경우 주석을 해제하여 먼저 설치: %pip install lightgbm scikit-learn pandas numpy
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore', category=FutureWarning)
root=Path('.')
for name in ['Faults.NNA', 'Faults27x7_var']:
    if not (root/name).exists():
        raise FileNotFoundError(f'{name} 파일을 노트북 실행 폴더에 넣어주세요. 현재 폴더: {root.resolve()}')
cols=[x.strip() for x in (root/'Faults27x7_var').read_text(encoding='utf-8').splitlines() if x.strip()]
raw=pd.read_csv(root/'Faults.NNA',sep=r'\s+',header=None,names=cols)
assert raw.shape==(1941,34), f'예상과 다른 데이터 모양: {raw.shape}'
encoder=LabelEncoder()
y_full=encoder.fit_transform(raw[cols[27:]].idxmax(axis=1))
X_full=raw.iloc[:,:27].drop(columns='TypeOfSteel_A300')
idx_dev,idx_test=train_test_split(np.arange(len(y_full)),test_size=0.2,stratify=y_full,random_state=42)
X_dev=X_full.iloc[idx_dev].reset_index(drop=True)
y_dev=y_full[idx_dev]
B=int(encoder.transform(['Bumps'])[0])
P=int(encoder.transform(['Pastry'])[0])
O=int(encoder.transform(['Other_Faults'])[0])
print('개발:',X_dev.shape,'| 보류 테스트(미사용):',len(idx_test))
print('클래스:',list(encoder.classes_))

개발: (1552, 26) | 보류 테스트(미사용): 389
클래스: ['Bumps', 'Dirtiness', 'K_Scatch', 'Other_Faults', 'Pastry', 'Stains', 'Z_Scratch']


In [2]:
# 기존 3~6단계와 동일한 베이스모델 설정
et_base=ExtraTreesClassifier(
    random_state=42,n_jobs=1,n_estimators=500,
    min_samples_split=5,min_samples_leaf=1,
    max_features=0.8,max_depth=None,
    class_weight='balanced_subsample'
)
lgb_base=LGBMClassifier(
    objective='multiclass',random_state=42,n_jobs=1,verbose=-1,
    subsample_freq=1,subsample=0.8,reg_lambda=0,
    num_leaves=22,n_estimators=700,min_child_samples=5,
    max_depth=-1,learning_rate=0.05,
    colsample_bytree=0.6,class_weight=None
)

# 이 실험에서 확인할 가중치는 평가 전 미리 정한 네 가지뿐입니다.
# 3단계의 class_weight='balanced'와 달리 전체가 아닌 특정 난제 클래스만 조금 조정합니다.
WEIGHT_PRESETS={
    '기존 LR C=1':{},
    'Pastry ×1.25':{'Pastry':1.25},
    'Pastry ×1.50':{'Pastry':1.50},
    'Bumps ×1.15 + Pastry ×1.25':{'Bumps':1.15,'Pastry':1.25}
}


def sample_weight_for(y, factors):
    result=np.ones(len(y),dtype=float)
    for name,weight in factors.items():
        target=int(encoder.transform([name])[0])
        result[y==target]=weight
    return result


def macro(y,p):
    return f1_score(y,p,average='macro',zero_division=0)

print('비교 후보:',list(WEIGHT_PRESETS))

비교 후보: ['기존 LR C=1', 'Pastry ×1.25', 'Pastry ×1.50', 'Bumps ×1.15 + Pastry ×1.25']


In [3]:
# [7-1] 같은 외부 Fold 안에서 네 가지 메타모델 비교
outer=StratifiedKFold(n_splits=3,shuffle=True,random_state=2069)
inner=StratifiedKFold(n_splits=5,shuffle=True,random_state=1)
rows=[]
collected={name:[] for name in ['정답',*WEIGHT_PRESETS]}

for fold,(tr,va) in enumerate(outer.split(X_dev,y_dev),1):
    Xa,Xv=X_dev.iloc[tr],X_dev.iloc[va]
    ya,yv=y_dev[tr],y_dev[va]
    print(f'외부 Fold {fold}: 내부 OOF 확률 생성 중...',flush=True)
    et_oof=cross_val_predict(clone(et_base),Xa,ya,cv=inner,method='predict_proba',n_jobs=1)
    lgb_oof=cross_val_predict(clone(lgb_base),Xa,ya,cv=inner,method='predict_proba',n_jobs=1)
    Z_train=np.hstack([et_oof,lgb_oof])
    et_valid=clone(et_base).fit(Xa,ya).predict_proba(Xv)
    lgb_valid=clone(lgb_base).fit(Xa,ya).predict_proba(Xv)
    Z_valid=np.hstack([et_valid,lgb_valid])
    r={'Fold':fold}
    collected['정답'].append(np.asarray(yv))
    for name,factors in WEIGHT_PRESETS.items():
        meta=LogisticRegression(C=1.0,max_iter=3000)
        meta.fit(Z_train,ya,sample_weight=sample_weight_for(ya,factors))
        pred=meta.predict(Z_valid)
        r[name]=macro(yv,pred)
        collected[name].append(pred)
    rows.append(r)
    print(f'Fold {fold} 완료 | 기준={r["기존 LR C=1"]:.4f} | ' +
          ' | '.join(f'{key}={r[key]:.4f}' for key in WEIGHT_PRESETS if key!='기존 LR C=1'),flush=True)

fold_scores=pd.DataFrame(rows)
print('\n[7단계 외부 Fold별 결과]')
display(fold_scores.round(4))
score_cols=list(WEIGHT_PRESETS)
summary=pd.DataFrame({
    '평균 Macro F1':fold_scores[score_cols].mean(),
    '표준편차':fold_scores[score_cols].std(ddof=0),
    '기준 대비':fold_scores[score_cols].mean()-fold_scores['기존 LR C=1'].mean()
}).sort_values('평균 Macro F1',ascending=False)
print('\n[같은 분할에서 모델 간 비교]')
display(summary.round(4))
print('목표 0.835 도달 후보:',list(summary.index[summary['평균 Macro F1']>=0.835]))
fold_scores.to_csv('stage7_fold_scores.csv',index=False,encoding='utf-8-sig')

외부 Fold 1: 내부 OOF 확률 생성 중...
Fold 1 완료 | 기준=0.8268 | Pastry ×1.25=0.8236 | Pastry ×1.50=0.8254 | Bumps ×1.15 + Pastry ×1.25=0.8238
외부 Fold 2: 내부 OOF 확률 생성 중...
Fold 2 완료 | 기준=0.8194 | Pastry ×1.25=0.8177 | Pastry ×1.50=0.8218 | Bumps ×1.15 + Pastry ×1.25=0.8191
외부 Fold 3: 내부 OOF 확률 생성 중...
Fold 3 완료 | 기준=0.8201 | Pastry ×1.25=0.8205 | Pastry ×1.50=0.8237 | Bumps ×1.15 + Pastry ×1.25=0.8221

[7단계 외부 Fold별 결과]


,Fold,기존 LR C=1,Pastry ×1.25,Pastry ×1.50,Bumps ×1.15 + Pastry ×1.25
0,1,0.8268,0.8236,0.8254,0.8238
1,2,0.8194,0.8177,0.8218,0.8191
2,3,0.8201,0.8205,0.8237,0.8221



[같은 분할에서 모델 간 비교]


,평균 Macro F1,표준편차,기준 대비
Pastry ×1.50,0.8236,0.0014,0.0015
기존 LR C=1,0.8221,0.0034,0.0000
Bumps ×1.15 + Pastry ×1.25,0.8217,0.0020,-0.0004
Pastry ×1.25,0.8206,0.0024,-0.0015


목표 0.835 도달 후보: []


In [4]:
# [7-2] 가장 잘 나온 방법의 각 클래스별 F1을 기준 모델과 비교
joined={name:np.concatenate(parts) for name,parts in collected.items()}
best=summary.index[0]
true=joined['정답']
labels=np.arange(len(encoder.classes_))
class_table=pd.DataFrame({
    '기존 LR C=1':f1_score(true,joined['기존 LR C=1'],average=None,labels=labels,zero_division=0),
    f'후보: {best}':f1_score(true,joined[best],average=None,labels=labels,zero_division=0)
},index=encoder.classes_)
class_table['변화']=class_table.iloc[:,1]-class_table.iloc[:,0]
print('[클래스별 F1 — 전체 외부 예측 연결, Fold 평균과 다름]')
display(class_table.round(4))
print('\n[선택된 후보 오분류 행렬]')
display(pd.DataFrame(confusion_matrix(true,joined[best],labels=labels),
                     index=encoder.classes_,columns=encoder.classes_))
print('\n변경 예측 수:',int(np.sum(joined['기존 LR C=1'] != joined[best])))
print('주의: 여기서 최고 후보를 고른 것은 개발 데이터 탐색입니다. 마지막 389행 독립 테스트에서는 후보 재선택을 하지 마세요.')

[클래스별 F1 — 전체 외부 예측 연결, Fold 평균과 다름]


,기존 LR C=1,후보: Pastry ×1.50,변화
Bumps,0.6602,0.6645,0.0043
Dirtiness,0.8462,0.8462,0.0000
K_Scatch,0.9660,0.9660,0.0000
Other_Faults,0.7504,0.7513,0.0009
Pastry,0.6579,0.6639,0.0060
Stains,0.9474,0.9474,0.0000
Z_Scratch,0.9356,0.9356,0.0000



[선택된 후보 오분류 행렬]


,Bumps,Dirtiness,K_Scatch,Other_Faults,Pastry,Stains,Z_Scratch
Bumps,205,0,0,106,10,0,0
Dirtiness,2,33,0,7,2,0,0
K_Scatch,1,0,298,12,0,2,0
Other_Faults,75,1,5,429,23,0,5
Pastry,12,0,0,34,80,0,0
Stains,1,0,0,3,0,54,0
Z_Scratch,0,0,1,13,0,0,138



변경 예측 수: 14
주의: 여기서 최고 후보를 고른 것은 개발 데이터 탐색입니다. 마지막 389행 독립 테스트에서는 후보 재선택을 하지 마세요.


In [5]:
print("[7단계 모델별 평균 성능]")
display(summary.round(4))

print("\n[Fold별 성능]")
display(fold_scores.round(4))

print("\n[기존 대비 최고 후보 개선량]")
print(
    round(
        summary.iloc[0]["평균 Macro F1"]
        - summary.loc["기존 LR C=1", "평균 Macro F1"],
        4
    )
)

[7단계 모델별 평균 성능]


,평균 Macro F1,표준편차,기준 대비
Pastry ×1.50,0.8236,0.0014,0.0015
기존 LR C=1,0.8221,0.0034,0.0000
Bumps ×1.15 + Pastry ×1.25,0.8217,0.0020,-0.0004
Pastry ×1.25,0.8206,0.0024,-0.0015



[Fold별 성능]


,Fold,기존 LR C=1,Pastry ×1.25,Pastry ×1.50,Bumps ×1.15 + Pastry ×1.25
0,1,0.8268,0.8236,0.8254,0.8238
1,2,0.8194,0.8177,0.8218,0.8191
2,3,0.8201,0.8205,0.8237,0.8221



[기존 대비 최고 후보 개선량]
0.0015
